# Lab 18-11 · Timing Leaks, Power Analysis, PUF Keys and Glitches

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import shutil
import subprocess
import tempfile
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import hwsec as hs

## Part 1 · The S-box

*Predict* S(0x00) and S(0x53) (FIPS 197's worked values).

In [2]:
sb = hs.aes_sbox()
print(f"S(0x00) = 0x{sb[0x00]:02X}, S(0x53) = 0x{sb[0x53]:02X}, permutation: {sorted(sb.tolist()) == list(range(256))}")

S(0x00) = 0x63, S(0x53) = 0xED, permutation: True


## Part 2 · Timing

*Predict* the queries for a 4-byte secret at 200 measurements per guess, and what happens with fewer.

In [3]:
secret = bytes([0x3A, 0x91, 0x07, 0xEE])
for reps in (200, 20, 5):
    got, q = hs.timing_attack(secret, reps=reps)
    print(f"{reps:3d} measurements per guess: recovered {got.hex()} ({'correct' if got == secret else 'wrong'}), {q} queries")
got, q = hs.timing_attack(secret, compare=hs.constant_time)
print(f"constant-time comparison: recovered {got.hex()} ({'correct' if got == secret else 'wrong'})")

200 measurements per guess: recovered 3a9107ee (correct), 153839 queries
 20 measurements per guess: recovered 4f9b0c00 (wrong), 15616 queries
  5 measurements per guess: recovered 3a656100 (wrong), 4096 queries
constant-time comparison: recovered acd3f600 (wrong)


## Part 3 · CPA

*Predict* the traces needed at noise 1, 3 and 6, then mask.

In [4]:
for s in (1, 3, 6):
    ranks = [hs.cpa_rank(*hs.simulate_traces(0x2B, n, s), 0x2B)[1] for n in (10, 30, 100, 300, 1000)]
    print(f"noise {s}: predicted {hs.traces_needed(hs.theoretical_rho(s)):6.0f} traces; rank of true key at 10/30/100/300/1000 traces: {ranks}")
p, l = hs.simulate_traces(0x2B, 20000, 1, masked=True)
print("masked, 20 000 traces:", hs.cpa_rank(p, l, 0x2B))

noise 1: predicted     30 traces; rank of true key at 10/30/100/300/1000 traces: [6, 0, 0, 0, 0]
noise 3: predicted    111 traces; rank of true key at 10/30/100/300/1000 traces: [102, 3, 0, 0, 0]
noise 6: predicted    385 traces; rank of true key at 10/30/100/300/1000 traces: [193, 51, 12, 1, 0]
masked, 20 000 traces: (214, 95, 0.0062980013964733786)


## Part 4 · PUF

*Predict* the repetition length for a 10⁻⁶ key failure at p = 0.05, 0.15 and 0.205.

In [5]:
ns = np.arange(1, 101, 2)
for p in (0.05, 0.15, 0.205):
    plt.semilogy(ns, [max(hs.key_failure(p, n), 1e-12) for n in ns], label=f"p = {p}")
    print(f"p = {p}: n = {hs.smallest_repetition(p, 1e-6)}")
plt.axhline(1e-6, color="grey", linestyle=":"); plt.xlabel("repetition length n"); plt.ylabel("128-bit key failure"); plt.legend(); plt.show()
print("p = 0.15, n = 9: formula", round(hs.key_failure(0.15, 9), 4), "simulation", round(hs.simulate_puf(0.15, 9), 4))

p = 0.05: n = 19
p = 0.15: n = 47
p = 0.205: n = 75


p = 0.15, n = 9: formula 0.5145 simulation 0.5107


## Part 5 · Glitches and the C comparison

In [6]:
print("single check:", hs.expected_attempts(1e-4) / 10, "s; double check:", hs.expected_attempts(1e-8) / 10 / 86400, "days")
cc = shutil.which("cc") or shutil.which("clang") or shutil.which("gcc")
if cc:
    with tempfile.TemporaryDirectory() as td:
        subprocess.run([cc, "-std=c17", "-O1", "-o", f"{td}/t", "c/ctcompare_test.c"], check=True)
        print(subprocess.run([f"{td}/t"], capture_output=True, text=True).stdout)

single check: 1000.0 s; double check: 115.74074074074075 days


20000 cases: constant-time always examines 16 bytes; early exit examines prefix + 1
TEST PASSED



## Discussion

The questions and worked answers are in the [handout](README.md#discussion).